# Probe Guidance: Predictor Evaluation

In [1]:
import sys
sys.path.insert(0, "/home/jack/code/vjepa2-probe-guidance/vjepa2")
print(sys.path)

['/home/jack/code/vjepa2-probe-guidance/vjepa2', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/home/jack/code/vjepa2-probe-guidance/.venv/lib/python3.10/site-packages', '/home/jack/code/vjepa2-probe-guidance/vjepa2/src', '/home/jack/code/vjepa2-probe-guidance/.venv/lib/python3.10/site-packages/rerun_sdk']


In [2]:
from pathlib import Path
import copy
import os

import numpy as np
import torch
from torch.nn import functional as F
import torchvision.transforms as T
from scipy.spatial.transform import Rotation
from tqdm import tqdm

In [3]:
from app.vjepa_ll_probe_guidance.ll_probe_guidance import LLProbeGuidanceDataset, standardize_actions, standardize_states
from app.vjepa_ll_probe_guidance.utils import init_video_model
from app.vjepa_ll_probe_guidance.transforms import make_transforms

/home/jack/code/vjepa2-probe-guidance/.venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Model Initialization

In [4]:
# TODO: initialize model from model config file.
encoder, predictor = init_video_model(
        device="cuda:0",
        patch_size=16,
        max_num_frames=512,
        tubelet_size=2,
        model_name="vit_large",
        crop_size=256,
        pred_depth=12,
        pred_num_heads=12,
        pred_embed_dim=768,
        action_embed_dim=6,
        predictor_type="ac",
        pred_is_frame_causal=True,
        use_extrinsics=False,
        use_sdpa=True,
        use_rope=True
    )
target_encoder = copy.deepcopy(encoder)

encoder.eval()
predictor.eval()
target_encoder.eval()

def load_state_dict_with_ddp_fix(model, state_dict):
    new_state_dict = {}
    for k, v in state_dict.items():
        # Remove 'module.' prefix if it exists
        new_key = k.replace("module.", "")
        new_state_dict[new_key] = v

    model.load_state_dict(new_state_dict, strict=True)
    return model

resume_path = os.path.join("/home/jack/code/vjepa2-probe-guidance/vjepa2/outputs/ll_probe_guidance_vitl_4", "best.pt")
if os.path.exists(resume_path):
    print(f"Loading checkpoint from {resume_path}")
    checkpoint = torch.load(resume_path, map_location=torch.device("cpu"))
    encoder = load_state_dict_with_ddp_fix(encoder, checkpoint["encoder"])
    predictor = load_state_dict_with_ddp_fix(predictor, checkpoint["predictor"])
    target_encoder = load_state_dict_with_ddp_fix(target_encoder, checkpoint["target_encoder"])
else:
    print(f"Checkpoint not found at {resume_path}")

print("=" * 20 + "PREDICTOR" + "=" * 20)
print(predictor)
print("=" * 20 + "TARGET ENCODER" + "=" * 20)
print(target_encoder)

Loading checkpoint from /home/jack/code/vjepa2-probe-guidance/vjepa2/outputs/ll_probe_guidance_vitl_4/best.pt
====================PREDICTOR====================
VisionTransformerPredictorAC(
  (predictor_embed): Linear(in_features=1024, out_features=768, bias=True)
  (action_encoder): Linear(in_features=6, out_features=768, bias=True)
  (state_encoder): Linear(in_features=6, out_features=768, bias=True)
  (extrinsics_encoder): Linear(in_features=5, out_features=768, bias=True)
  (predictor_blocks): ModuleList(
    (0-11): 12 x ACBlock(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
      (attn): ACRoPEAttention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, b

## Dataset Initialization

In [5]:
crop_size = 256
tokens_per_frame = int((crop_size // encoder.patch_size) ** 2)
transform = make_transforms(
    crop_size=crop_size,
)

# SET THIS TO THE NUMBER OF FRAMES YOU WANT IN A CLIP
T = 8

dataset = LLProbeGuidanceDataset(
    data_root="/home/jack/data/probe_guidance_dataset_june/test",
    frames_per_clip=T,
    frame_skip=1,
    frames_per_second=4,
    transform=transform,
    is_train=False,
)

loader = torch.utils.data.DataLoader(
    dataset,
    shuffle=False,
    batch_size=8,
    drop_last=True,
    pin_memory=False,
    num_workers=8,
)

Scanning 7 episodes for valid tracking clips...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 37.85it/s]

Retained 7 episodes.


In [6]:
def load_clips(sample):
    clips = sample[0].to("cuda:0", non_blocking=True)  # [B C T H W]
    actions = sample[1]  # [B T-1 6]
    states = sample[2]  # [B T 6]
    extrinsics = sample[3].to("cuda:0", dtype=torch.float, non_blocking=True)  # [B T 6]
    return (clips, actions, states, extrinsics)

sample = next(iter(loader))
clips, actions, states, _ = load_clips(sample)
clips = clips[:, :, :T]
actions = actions[:, :T-1]
states = states[:, :T]
print(f"clips: {clips.shape}; states: {states.shape}; actions: {actions.shape}")

clips: torch.Size([1, 3, 8, 256, 256]); states: torch.Size([1, 8, 6]); actions: torch.Size([1, 7, 6])


## Test Set Evaluation

In [9]:
def forward_target(c, normalize_reps=True):
    B, C, T, H, W = c.size()
    c = c.permute(0, 2, 1, 3, 4).flatten(0, 1).unsqueeze(2).repeat(1, 1, 2, 1, 1)
    h = encoder(c)
    h = h.view(B, T, -1, h.size(-1)).flatten(1, 2)
    if normalize_reps:
        h = F.layer_norm(h, (h.size(-1),))
    return h

def step_predictor(z, a, s, normalize_reps=True):
    standardized_actions = standardize_actions(a).to("cuda:0", dtype=torch.float, non_blocking=True)
    standardized_states = standardize_states(s).to("cuda:0", dtype=torch.float, non_blocking=True)
    z = predictor(z, standardized_actions, standardized_states)[:, -tokens_per_frame:]
    if normalize_reps:
        z = F.layer_norm(z, (z.size(-1),))
    return z

def loss_fn(z, h):
    # NOTE: The caller is now responsible for making sure that the tokens in z and h align.
    # TODO: MAKE SURE THIS IS UPDATED IN ENERGY LANDSCAPE SECTION AND BEYOND (world model section)
    loss = torch.abs(z - h)  # [B, N, D]
    loss = torch.mean(loss, dim=[1, 2])
    return loss.tolist()

In [11]:
# NOTE: the actions are in the probe's coordinate frame, while the states are 
# in the camera's coordinate frame. Does this have a negative impact on the model?
# Nonetheless, computing the new pose needs to account for this discrepancy.

# NOTE: state and action may be standardized, may need to account for this or rethink dataset/dataloader
# to not do standardization. Perhaps standardization should happen in training/testing and before passing to model
# forward functions (or within the forward functions themselves). doing it in dataset makes it hard to do visualizations.
# breaking standardization out can be good from a single-responsibility principle standpoint.

def compute_new_pose(pose, action):
    """
    :param pose: [B, T=1, 6] (x, y, z, rx, ry, rz in Euler angles) in camera frame
    :param action: [B, T=1, 6] (dx, dy, dz, drx, dry, drz) in local object frame
    :returns: [B, T=1, 6] updated pose in camera frame
    """
    device, dtype = pose.device, pose.dtype
    
    pose_np = pose[:, 0].detach().cpu().numpy()
    action_np = action[:, 0].detach().cpu().numpy()

    R_pose = Rotation.from_euler("xyz", pose_np[:, 3:6], degrees=True)
    R_pose_mat = R_pose.as_matrix()  # [B, 3, 3]

    # local action translation: [B, 3, 1] column vector
    local_dxyz = action_np[:, :3, None]
    
    global_dxyz = (R_pose_mat @ local_dxyz).squeeze(-1)  # [B, 3]
    new_xyz = pose_np[:, :3] + global_dxyz

    R_action = Rotation.from_euler("xyz", action_np[:, 3:6], degrees=True)
    R_new = R_pose * R_action

    new_angle = R_new.as_euler("xyz", degrees=True)  # [B, 3]

    new_pose = np.concatenate([new_xyz, new_angle], axis=-1)

    # Restore dimensions to [B, T=1, 6]
    return torch.from_numpy(new_pose).to(device=device, dtype=dtype)[:, None]